# Atlanta Mobility CUDA readiness and benchmark
Run in Google Colab with **Runtime → Change runtime type → T4 GPU**. The session is ephemeral: retain the receipt before it ends.

In [1]:
!git clone https://github.com/triasha72/atlanta-mobility-resilience-digital-twin.git || (cd atlanta-mobility-resilience-digital-twin && git pull)
%cd /content/atlanta-mobility-resilience-digital-twin
%pip install -q -e '.[dev,gnn]'
!nvidia-smi || echo 'No NVIDIA GPU is attached. Select Runtime → Change runtime type → T4 GPU, then reconnect and rerun.'

Cloning into 'atlanta-mobility-resilience-digital-twin'...
remote: Enumerating objects: 439, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (262/262), done.
remote: Total 439 (delta 237), reused 344 (delta 150), pack-reused 0 (from 0)
Receiving objects: 100% (439/439), 251.62 KiB | 1.95 MiB/s, done.
Resolving deltas: 100% (237/237), done.
/content/atlanta-mobility-resilience-digital-twin
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.0 MB/s eta 0:00:00
  Building editable for atlanta-mobility-res

In [2]:
!PYTHONPATH=src python scripts/probe_cuda_runtime.py --output reports/colab_gpu_probe.json
!cat reports/colab_gpu_probe.json

{
  "schema_version": "1.0",
  "captured_at": "2026-09-13T01:37:35.777513+00:00",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_available": false,
  "cuda_runtime_available": true,
  "nvidia_smi": "Tesla T4, 580.82.07, 15360 MiB",
  "torch_available": true,
  "torch_version": "2.11.0+cu128",
  "torch_cuda_available": true,
  "torch_cuda_version": "12.8",
  "torch_device_name": "Tesla T4"
}
{
  "schema_version": "1.0",
  "captured_at": "2026-09-13T01:37:35.777513+00:00",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_available": false,
  "cuda_runtime_available": true,
  "nvidia_smi": "Tesla T4, 580.82.07, 15360 MiB",
  "torch_available": true,
  "torch_version": "2.11.0+cu128",
  "torch_cuda_available": true,
  "torch_cuda_version": "12.8",
  "torch_device_name": "Tesla T4"
}


In [3]:
# Rebuild ignored public inputs and the tract graph in this fresh runtime.
!PYTHONPATH=src python scripts/materialize_acs_origins.py
!PYTHONPATH=src python scripts/materialize_osm_destinations.py
!PYTHONPATH=src python scripts/benchmark_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cpu_benchmark.json
!cat reports/colab_cpu_benchmark.json

{
  "dataset": "American Community Survey tract estimates and Census Gazetteer",
  "acs_release": "acs2024_5yr",
  "acs_years": "2020-2024",
  "tract_count": 50,
  "represented_population_estimate": 216659,
  "population_moe_median": 642.5,
  "median_household_income_median": 108544.5,
  "source_hashes": {
    "fulton_acs.json": "3dbbb0aab62b834d9dd987070ea7c6002b1997c311f2d4f6651049f6ea5230b7",
    "dekalb_acs.json": "a4b8251222d8f02fa4549bc558925baccc4701fea5850da7faa8faa38cf74868",
    "2024_gaz_tracts_13.txt": "abfb7592f637df78ea99837329fcf5ef3d59921e3913b5fec35dfa4781095176"
  },
  "interpretation": "ACS values are survey estimates with margins of error; representative tract coordinates are routing origins, not observed trip starts.",
  "sources": {
    "fulton_acs.json": "https://api.censusreporter.org/1.0/data/show/acs2024_5yr?table_ids=B01003,B19013&geo_ids=140%7C05000US13121",
    "dekalb_acs.json": "https://api.censusreporter.org/1.0/data/show/acs2024_5yr?table_ids=B01003,B19

A CUDA-capable receipt is necessary before adding cuGraph results. If `cugraph_available` is false, preserve the receipt and do not label the benchmark GPU-accelerated.

In [5]:
!nvidia-smi
PYTHONPATH=src python scripts/probe_cuda_runtime.py --output reports/colab_gpu_probe.json

SyntaxError: invalid syntax (54734388.py, line 2)

In [6]:
%cd /content/atlanta-mobility-resilience-digital-twin
!git pull --ff-only
!git rev-parse --short HEAD
!test -f scripts/benchmark_cugraph_routing.py && echo "GPU benchmark script present"


/content/atlanta-mobility-resilience-digital-twin
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 15 (delta 7), reused 11 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 7.58 KiB | 1.89 MiB/s, done.
From https://github.com/triasha72/atlanta-mobility-resilience-digital-twin
   714ac94..336fb5c  main       -> origin/main
Updating 714ac94..336fb5c
Fast-forward
 notebooks/colab_cuda_benchmark.ipynb | 40 ++++++++++++++----
 scripts/benchmark_cugraph_routing.py | 79 ++++++++++++++++++++++++++++++++++++
 src/amrdt/gpu.py                     | 24 +++++++++++
 tests/test_gpu.py                    | 26 ++++++++++++
 4 files changed, 161 insertions(+), 8 deletions(-)
 create mode 100644 scripts/benchmark_cugraph_routing.py
 create mode 100644 src/amrdt/gpu.py
 create mode 100644 tests/test_gpu.py
336fb5c
GPU benchmark script present


In [16]:
# Cell 1 — install project and confirm T4
%cd /content
!git clone https://github.com/triasha72/atlanta-mobility-resilience-digital-twin.git || (cd atlanta-mobility-resilience-digital-twin && git pull --ff-only)
%cd /content/atlanta-mobility-resilience-digital-twin
%pip install -q -e '.[dev,gnn]'
!nvidia-smi

/content
fatal: destination path 'atlanta-mobility-resilience-digital-twin' already exists and is not an empty directory.
Already up to date.
/content/atlanta-mobility-resilience-digital-twin
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for atlanta-mobility-resilience-digital-twin (pyproject.toml) ... done
Sun Sep 13 02:03:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                

In [17]:
# Cell 2 — install cuGraph/RAPIDS
!git clone --depth 1 https://github.com/rapidsai/rapidsai-csp-utils.git || git -C rapidsai-csp-utils pull --ff-only
!python rapidsai-csp-utils/colab/pip-install.py
!python -c "import cugraph; print('cuGraph', cugraph.__version__)"

fatal: destination path 'rapidsai-csp-utils' already exists and is not an empty directory.
Already up to date.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.13.15 environment at: /usr
Checked 11 packages in 98ms

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do not run any further installation from the conda based installation methods, as they may cause issues!

        Please ensure that you're pulling from the git repo to remain updated with the latest working install scripts.

        Troubleshooting:
            - If there is an installation failure, please check back on RAPIDSAI owned templates/notebooks to see how to update your personal files.
            - If an installation failure persists when using the latest script, please make an issue on https://github.com/rapidsai-community/rapidsai-csp-utils
        **************************************************************

In [18]:
# Cell 3 — GPU receipt
!PYTHONPATH=src python scripts/probe_cuda_runtime.py --output reports/colab_gpu_probe.json
!cat reports/colab_gpu_probe.json

{
  "schema_version": "1.0",
  "captured_at": "2026-09-13T02:04:11.170468+00:00",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_available": true,
  "cuda_runtime_available": true,
  "nvidia_smi": "Tesla T4, 580.82.07, 15360 MiB",
  "torch_available": true,
  "torch_version": "2.11.0+cu128",
  "torch_cuda_available": true,
  "torch_cuda_version": "12.8",
  "torch_device_name": "Tesla T4"
}
{
  "schema_version": "1.0",
  "captured_at": "2026-09-13T02:04:11.170468+00:00",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_available": true,
  "cuda_runtime_available": true,
  "nvidia_smi": "Tesla T4, 580.82.07, 15360 MiB",
  "torch_available": true,
  "torch_version": "2.11.0+cu128",
  "torch_cuda_available": true,
  "torch_cuda_version": "12.8",
  "torch_device_name": "Tesla T4"
}


In [19]:
# Cell 4 — rebuild inputs and CPU comparison benchmark
!PYTHONPATH=src python scripts/materialize_acs_origins.py
!PYTHONPATH=src python scripts/materialize_osm_destinations.py
!PYTHONPATH=src python scripts/benchmark_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cpu_benchmark.json
!cat reports/colab_cpu_benchmark.json

{
  "dataset": "American Community Survey tract estimates and Census Gazetteer",
  "acs_release": "acs2024_5yr",
  "acs_years": "2020-2024",
  "tract_count": 50,
  "represented_population_estimate": 216659,
  "population_moe_median": 642.5,
  "median_household_income_median": 108544.5,
  "source_hashes": {
    "fulton_acs.json": "3dbbb0aab62b834d9dd987070ea7c6002b1997c311f2d4f6651049f6ea5230b7",
    "dekalb_acs.json": "a4b8251222d8f02fa4549bc558925baccc4701fea5850da7faa8faa38cf74868",
    "2024_gaz_tracts_13.txt": "abfb7592f637df78ea99837329fcf5ef3d59921e3913b5fec35dfa4781095176"
  },
  "interpretation": "ACS values are survey estimates with margins of error; representative tract coordinates are routing origins, not observed trip starts.",
  "sources": {
    "fulton_acs.json": "https://api.censusreporter.org/1.0/data/show/acs2024_5yr?table_ids=B01003,B19013&geo_ids=140%7C05000US13121",
    "dekalb_acs.json": "https://api.censusreporter.org/1.0/data/show/acs2024_5yr?table_ids=B01003,B19

In [20]:
# Cell 5 — cuGraph GPU routing benchmark
!PYTHONPATH=src python scripts/benchmark_cugraph_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cugraph_benchmark.json
!cat reports/colab_cugraph_benchmark.json

{
  "schema_version": "1.0",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_version": "26.02.00",
  "nodes": 22145,
  "input_edges": 56516,
  "coalesced_edges": 55512,
  "vertex_ids_renumbered": false,
  "origins": 50,
  "destinations": 101,
  "od_pairs": 5050,
  "reachable_od_pairs": 4700,
  "gpu_graph_setup_seconds": 0.5418011309998292,
  "gpu_routing_seconds": 9.582007608999902
}
{
  "schema_version": "1.0",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_version": "26.02.00",
  "nodes": 22145,
  "input_edges": 56516,
  "coalesced_edges": 55512,
  "vertex_ids_renumbered": false,
  "origins": 50,
  "destinations": 101,
  "od_pairs": 5050,
  "reachable_od_pairs": 4700,
  "gpu_graph_setup_seconds": 0.5418011309998292,
  "gpu_routing_seconds": 9.582007608999902
}


In [12]:
%cd /content/atlanta-mobility-resilience-digital-twin
!git pull --ff-only
!git rev-parse --short HEAD

/content/atlanta-mobility-resilience-digital-twin
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 653 bytes | 653.00 KiB/s, done.
From https://github.com/triasha72/atlanta-mobility-resilience-digital-twin
   336fb5c..848fa49  main       -> origin/main
Updating 336fb5c..848fa49
Fast-forward
 scripts/benchmark_cugraph_routing.py | 12 +++++++++++-
 1 file changed, 11 insertions(+), 1 deletion(-)
848fa49


In [14]:
!PYTHONPATH=src python scripts/benchmark_cugraph_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cugraph_benchmark.json
!cat reports/colab_cugraph_benchmark.json

Traceback (most recent call last):
  File "/content/atlanta-mobility-resilience-digital-twin/scripts/benchmark_cugraph_routing.py", line 89, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/atlanta-mobility-resilience-digital-twin/scripts/benchmark_cugraph_routing.py", line 62, in main
    distances = cugraph.sssp(gpu_graph, source=int(source), weight="weight")
TypeError: sssp() got an unexpected keyword argument 'weight'
cat: reports/colab_cugraph_benchmark.json: No such file or directory


In [15]:
%cd /content/atlanta-mobility-resilience-digital-twin
!git pull --ff-only
!git rev-parse --short HEAD
!PYTHONPATH=src python scripts/benchmark_cugraph_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cugraph_benchmark.json
!cat reports/colab_cugraph_benchmark.json


/content/atlanta-mobility-resilience-digital-twin
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 481 bytes | 481.00 KiB/s, done.
From https://github.com/triasha72/atlanta-mobility-resilience-digital-twin
   848fa49..1ca7ff5  main       -> origin/main
Updating 848fa49..1ca7ff5
Fast-forward
 scripts/benchmark_cugraph_routing.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)
1ca7ff5
{
  "schema_version": "1.0",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cugraph_version": "26.02.00",
  "nodes": 22145,
  "input_edges": 56516,
  "coalesced_edges": 55512,
  "vertex_ids_renumbered": false,
  "origins": 50,
  "destinations": 101,
  "od_pairs": 5050,
  "reachable_od_pairs": 4700,
  "gpu_graph_setup_seconds": 0.5363223869999274,
  "gpu_routing_seconds": 9.520969267000055
}
{
  "schema_version"